# Random Forest Regression — California Housing Dataset

This notebook trains and evaluates a Random Forest Regressor to predict median house values from the California Housing dataset.

**Model:** RandomForestRegressor  
**Dataset:** California Housing (sklearn) — 20,640 samples, 8 numeric features  
**Target:** Median house value (in 100k USD)  
**Deployment:** Amazon SageMaker (sklearn 1.2-1 container)

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

print(f'Dataset shape: {X.shape}')
print(f'Features: {list(X.columns)}')
print(f'Target: {housing.target_names[0] if hasattr(housing, "target_names") else "MedHouseVal"}')
X.head()

## 2. Exploratory Data Analysis

In [ ]:
df = pd.concat([X, y.rename('MedHouseVal')], axis=1)
print('Summary statistics:')
df.describe().round(3)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y, bins=50, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Median House Value (100k USD)')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution')

# Correlation heatmap
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], linewidths=0.5)
axes[1].set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Geographic distribution
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X['Longitude'], X['Latitude'], c=y, cmap='viridis',
                       alpha=0.3, s=1)
plt.colorbar(scatter, label='Median House Value (100k USD)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('California Housing: Geographic Distribution of House Values')
plt.tight_layout()
plt.show()

## 3. Train / Test Split + Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

## 4. Model Training

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1,
    )),
])

print('Training Random Forest Regressor...')
pipeline.fit(X_train, y_train)
print('Done!')

## 5. Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('=== Test Set Metrics ===')
print(f'RMSE : {rmse:.4f} (100k USD)  →  ${rmse * 100000:,.0f}')
print(f'MAE  : {mae:.4f}  (100k USD)  →  ${mae  * 100000:,.0f}')
print(f'R²   : {r2:.4f}')

In [ ]:
# Actual vs predicted plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, y_pred, alpha=0.2, color='#3b82f6', s=5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
              'r--', linewidth=2)
axes[0].set_xlabel('Actual (100k USD)')
axes[0].set_ylabel('Predicted (100k USD)')
axes[0].set_title(f'Actual vs. Predicted  (R² = {r2:.4f})')

residuals = y_test - y_pred
axes[1].hist(residuals, bins=60, color='#a78bfa', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (100k USD)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
rf = pipeline.named_steps['model']
importances = pd.Series(rf.feature_importances_, index=housing.feature_names)
importances = importances.sort_values(ascending=True)

importances.plot(kind='barh', figsize=(8, 5), color='#3b82f6')
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False).to_string())

## 7. Save Model Artifact

In [ ]:
OUT_DIR = Path('../ml/regression')
OUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUT_DIR / 'model.joblib'
joblib.dump(pipeline, model_path)
print(f'Model saved → {model_path}')

metrics = {
    'model_type': 'RandomForestRegressor',
    'dataset': 'California Housing (sklearn)',
    'n_train': len(X_train),
    'n_test': len(X_test),
    'rmse': float(rmse),
    'mae':  float(mae),
    'r2':   float(r2),
    'feature_names': list(housing.feature_names),
    'feature_importance': dict(sorted(
        zip(housing.feature_names, rf.feature_importances_.tolist()),
        key=lambda kv: kv[1], reverse=True
    )),
}
(OUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'Metrics saved → {OUT_DIR / "metrics.json"}')